01_data_preparation


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import pyreadstat

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

DATA_PATH = Path(r"D:\Sem 5\ML\Grp Assignment\PERV MER COPY - Copy.sav")   # the updated .sav
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

# Numbers from the action plan, used only as checks
EXPECTED = {"women": 135_825, "lfp": 47_936, "working": 44_491, "seeking": 3_445,
            "outside": 87_889, "trad": 35_231, "startup": 7_045, "gig": 2_215}

def check(name, got, key):
    flag = "OK" if got == EXPECTED[key] else "<<< DIFFERENT"
    print(f"{name}: {got:,} (plan: {EXPECTED[key]:,}) {flag}")

In [ ]:
df, meta = pyreadstat.read_sav(DATA_PATH)
print("Raw shape:", df.shape)

# Blanks are stored as empty text, not NaN
df = df.replace("", np.nan)

df = df.rename(columns={
    "Socia_Security": "prod_usage",
    "VAR00001": "ent_type_num",
    "status": "usual_status",
})

# Keep the SPSS value labels for later tables and plots
pd.to_pickle(meta.variable_value_labels, OUT_DIR / "value_labels.pkl")

In [ ]:
required = [
    "HHID", "state_perv1", "distcode_perv1", "b1q3_perv1",
    "b4q5_perv1", "b4q6_perv1", "b4q7_perv1", "b4q8_perv1", "b4q10_perv1",
    "b3q1_hhv1", "b3q3_hhv1", "b3q4_hhv1", "b3q5pt6_hhv1",
    "usual_status", "economy_type", "emp_st",
]
missing_cols = [c for c in required if c not in df.columns]
assert not missing_cols, f"Missing columns: {missing_cols}"
print("Persons:", len(df), "| Households:", df["HHID"].nunique())

# Code columns as strings so comparisons like == "2" work
for c in ["b4q5_perv1", "b4q7_perv1", "b4q8_perv1", "b1q3_perv1",
          "b3q3_hhv1", "b3q4_hhv1", "usual_status"]:
    df[c] = df[c].astype("string")
df["state_perv1"] = df["state_perv1"].astype(str)
df["distcode_perv1"] = df["distcode_perv1"].astype(str)

# Weights, if present
WEIGHT_COLS = [c for c in ["NSS_perv1", "NSC_perv1", "mult_perv1"] if c in df.columns]
print("Weight columns found:", WEIGHT_COLS)

In [ ]:
WORKING_CODES = ["11", "12", "21", "31", "41", "51"]
SEEKING_CODES = ["81"]

df["lfp"] = df["usual_status"].isin(WORKING_CODES + SEEKING_CODES).astype(int)
df["lfp_group"] = np.select(
    [df["usual_status"].isin(WORKING_CODES).to_numpy(),
     df["usual_status"].isin(SEEKING_CODES).to_numpy()],
    ["working", "seeking"], default="outside")

is_woman_15_59 = (df["b4q5_perv1"] == "2") & df["b4q6_perv1"].between(15, 59)
w = df[is_woman_15_59]
check("Women 15-59", len(w), "women")
check("In labour force", int(w["lfp"].sum()), "lfp")
check("Working", int((w["lfp_group"] == "working").sum()), "working")
check("Seeking", int((w["lfp_group"] == "seeking").sum()), "seeking")
check("Outside LF", int((w["lfp_group"] == "outside").sum()), "outside")

In [ ]:
df["age"] = df["b4q6_perv1"]
df["age_sq"] = df["age"] ** 2
df["edu_years"] = df["b4q10_perv1"]

# The supplied Edu_level is mis-coded, so rebuild it from b4q8_perv1
EDU_MAP = {
    "01": "1_illiterate_informal", "02": "1_illiterate_informal",
    "03": "1_illiterate_informal", "04": "1_illiterate_informal",
    "05": "2_primary_or_below",    "06": "2_primary_or_below",
    "07": "3_middle",              "08": "4_secondary",
    "10": "5_higher_secondary",    "11": "6_diploma_certificate",
    "12": "7_graduate",            "13": "8_postgraduate",
}
df["edu_group"] = df["b4q8_perv1"].map(EDU_MAP)
df["marital"] = df["b4q7_perv1"]

print("Unmapped education codes:", int(df["edu_group"].isna().sum()))
# Sanity check: mean years of education should rise down this table
print(df[is_woman_15_59].groupby("edu_group")["edu_years"].agg(["mean", "size"]).round(1))

In [ ]:
df["sector"] = df["b1q3_perv1"]            # 1 rural, 2 urban
df["state"] = df["state_perv1"]
df["religion"] = df["b3q3_hhv1"]
df["social_group"] = df["b3q4_hhv1"]
df["hh_size"] = df["b3q1_hhv1"]

# Per-capita consumption, not Hh_Total_Income_sum
pc_exp = df["b3q5pt6_hhv1"] / df["b3q1_hhv1"]
print("Non-positive per-capita expenditure:", int((pc_exp <= 0).sum()))
df["log_pc_exp"] = np.log(pc_exp.clip(lower=1))

In [ ]:
age = df["b4q6_perv1"]
df["_kid_u6"] = (age < 6).astype(int)
df["_elder60"] = (age >= 60).astype(int)
df["_man_15_59"] = ((df["b4q5_perv1"] == "1") & age.between(15, 59)).astype(int)

hh_comp = df.groupby("HHID")[["_kid_u6", "_elder60", "_man_15_59"]].sum()
hh_comp.columns = ["hh_kids_u6", "hh_elders_60plus", "hh_adult_men"]
df = df.join(hh_comp, on="HHID")

# Highest education among the OTHER household members (not the woman herself)
edu = df["edu_years"].fillna(0)
srt = pd.DataFrame({"HHID": df["HHID"], "edu": edu}).sort_values(
    ["HHID", "edu"], ascending=[True, False])
rank = srt.groupby("HHID").cumcount()
top1 = srt[rank == 0].set_index("HHID")["edu"]
top2 = srt[rank == 1].set_index("HHID")["edu"]
df["_top1"] = df["HHID"].map(top1)
df["_top2"] = df["HHID"].map(top2).fillna(0)
df["hh_max_edu_others"] = np.where(edu == df["_top1"], df["_top2"], df["_top1"])

In [ ]:
women = df[is_woman_15_59].copy()
women["own_economy_type"] = women["economy_type"].map(
    {0: "Traditional", 1: "Gig-proxy", 2: "Startup-proxy", 3: "Not working"})

working = women["lfp_group"] == "working"
et = women["economy_type"]

# Stage 2 targets (defined for working women only; NaN elsewhere)
women["y_gig_vs_trad"] = np.where(working & (et == 1), 1.0,
                          np.where(working & (et == 0), 0.0, np.nan))      # main: startup set aside
women["y_gig_vs_other"] = np.where(working & et.isin([0, 1, 2]),
                                   (et == 1).astype(float), np.nan)        # robustness
women["y_gigstartup_vs_trad"] = np.where(working & et.isin([0, 1, 2]),
                                         et.isin([1, 2]).astype(float), np.nan)  # sensitivity

print(pd.crosstab(women["lfp_group"], women["own_economy_type"], margins=True))
check("Traditional (working)", int((working & (et == 0)).sum()), "trad")
check("Startup-proxy (working)", int((working & (et == 2)).sum()), "startup")
check("Gig-proxy (working)", int((working & (et == 1)).sum()), "gig")

In [ ]:
FEATURES_CAT = ["marital", "edu_group", "sector", "state", "religion", "social_group"]
FEATURES_NUM = ["age", "age_sq", "edu_years", "hh_size", "log_pc_exp",
                "hh_kids_u6", "hh_elders_60plus", "hh_adult_men", "hh_max_edu_others"]
FEATURES_NO_STATE = [c for c in FEATURES_CAT if c != "state"]   # for the residual map

# Never features: own work, status, occupation, industry, enterprise, contract, social security, income
LEAKY = ["usual_status", "emp_st", "economy_type", "own_economy_type", "lfp", "lfp_group",
         "type_econ", "trad_econ", "gig_econ", "startup", "earning_mem", "Casual_Labour",
         "Total_Income", "Total_income_clean", "Inc_Cat", "Individual_Precarity",
         "Hh_Total_Income_sum", "Enterprise_Class", "ent_type_num", "Social_Security",
         "prod_usage", "b6q5_perv1", "b6q6_perv1", "b6q7_perv1", "b6q9_perv1", "b6q10_perv1",
         "b5pt1q5_perv1", "b5pt1q6_perv1", "b5pt1q9_perv1", "b5pt1q10_perv1", "b5pt1q11_perv1",
         "y_gig_vs_trad", "y_gig_vs_other", "y_gigstartup_vs_trad"]
overlap = set(FEATURES_CAT + FEATURES_NUM) & set(LEAKY)
assert not overlap, f"Leakage columns in feature list: {overlap}"

# Columns kept ONLY for analysis (Door or Trap, exploratory plots), never as features
ANALYSIS_ONLY = [c for c in [
    "Total_Income", "Total_income_clean", "Inc_Cat", "Individual_Precarity", "earning_mem",
    "Casual_Labour", "Social_Security", "b5pt1q11_perv1", "b5pt1q5_perv1", "b5pt1q6_perv1",
    "Enterprise_Class", "b3q5pt6_hhv1", "b3q1_hhv1"] if c in women.columns]
ANALYSIS_ONLY += [c for c in women.columns if c.startswith("b6q9_3pt")]   # day-wise casual wages
print("Day-wise wage columns found:", [c for c in women.columns if c.startswith("b6q9_3pt")])

miss = women[FEATURES_CAT + FEATURES_NUM].isna().mean().round(4)
print("Missing share per feature:")
print(miss[miss > 0] if (miss > 0).any() else "None")

In [ ]:
ID_COLS = ["HHID", "distcode_perv1", "lfp", "lfp_group", "usual_status", "emp_st",
           "economy_type", "own_economy_type", "y_gig_vs_trad", "y_gig_vs_other", "y_gigstartup_vs_trad"]
FEATURES = FEATURES_CAT + FEATURES_NUM

all_cols = list(dict.fromkeys(ID_COLS + WEIGHT_COLS + FEATURES + ANALYSIS_ONLY))
stage1 = women[all_cols].reset_index(drop=True)                      # all women
stage2 = stage1[stage1["lfp_group"] == "working"].reset_index(drop=True)   # working women
seeking = stage1[stage1["lfp_group"] == "seeking"].reset_index(drop=True)

# Outside the labour force: features only, no job, income or Stage 2 target columns
OUTSIDE_COLS = ["HHID", "distcode_perv1", "lfp", "lfp_group"] + WEIGHT_COLS + FEATURES
outside = stage1.loc[stage1["lfp_group"] == "outside", OUTSIDE_COLS].reset_index(drop=True)

stage1.to_pickle(OUT_DIR / "stage1_all_women.pkl")
stage2.to_pickle(OUT_DIR / "stage2_working_women.pkl")
outside.to_pickle(OUT_DIR / "outside_lf_women.pkl")
seeking.to_pickle(OUT_DIR / "seeking_women.pkl")
pd.Series({"FEATURES_CAT": FEATURES_CAT, "FEATURES_NUM": FEATURES_NUM,
           "FEATURES_NO_STATE": FEATURES_NO_STATE, "WEIGHT_COLS": WEIGHT_COLS,
           "ANALYSIS_ONLY": ANALYSIS_ONLY}).to_pickle(OUT_DIR / "feature_lists.pkl")

print("Stage 1:", stage1.shape, "| Stage 2:", stage2.shape,
      "| Outside LF:", outside.shape, "| Seeking:", seeking.shape)
check("Stage 2 rows", len(stage2), "working")
check("Outside rows", len(outside), "outside")
print("Stage 2 target counts (gig vs trad):", stage2["y_gig_vs_trad"].value_counts(dropna=False).to_dict())